<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CHAPTER_3_RAG_Part_II_Chatting_with_Your_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -qU supabase langchain-community
!pip install -qU langchain-groq langchain-text-splitters langchain-core
!pip install -qU langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.8 MB/s eta 0:00:00


In [27]:
import os
from google.colab import userdata
from supabase.client import create_client, Client
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_huggingface import HuggingFaceEmbeddings # Updated import
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

# ==========================================
# 1단계: 환경 변수 및 인증 정보 설정
# ==========================================
# 1) Supabase 접속 정보 (앞서 확인한 API 주소와 anon 키)
SUPABASE_URL = "https://tllrshlupjcikzaywaqm.supabase.co"
SUPABASE_KEY = userdata.get('supabase')

supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 2) Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

# ==========================================
# 2단계: 데이터 로드, 분할 및 임베딩 (Ingestion)
# ==========================================
# 문서 로드 및 분할
raw_documents = TextLoader('./test.txt', encoding='utf-8').load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

# 수파베이스와 연동할 HuggingFace 384차원 임베딩 모델 설정
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Supabase VectorStore에 데이터 Store (인덱싱 및 저장)
db = SupabaseVectorStore.from_documents(
    documents=documents,
    embedding=embeddings_model,
    client=supabase_client,
    table_name="documents",
    query_name="match_documents"  # SQL Editor에 등록한 함수 이름
)

# ==========================================
# 3단계: 검색기(Retriever) 및 프롬프트, Groq LLM 설정
# ==========================================
# 유사한(cosine similarity) 문서 2개를 뽑아오는 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k": 2})

query = 'Who are the key figures in the ancient greek history of philosophy?'

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}

    Question: {question}"""
)

# 초고속 가성비 모델인 Groq(Llama 3) 모델로 교체
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)


# ==========================================
# 4단계: LCEL @chain 데코레이터를 활용한 효율적인 파이썬 함수 캡슐화 (Generation)
# ==========================================
print("Running RAG system using Groq and Supabase...\n")

@chain
def qa_chain(input_query):
    # 1. Retrieval: 질문과 관련된 문서 조각 가져오기
    docs = retriever.invoke(input_query)

    # 2. Prompt 조립: 질문과 컨텍스트 결합
    formatted = prompt.invoke({"context": docs, "question": input_query})

    # 3. Generation: Groq LLM을 통해 답변 생성
    answer = llm.invoke(formatted)
    return answer

# RAG 체인 실행 및 결과 출력
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running RAG system using Groq and Supabase...

[최종 답변 결과]
Based on the provided context, the key figures mentioned in the history of ancient Greek philosophy are:

1. Thales of Miletus: Proposed that water was the fundamental substance (arche) underlying all matter.
2. Anaximander: Introduced the concept of the apeiron (infinite) as the origin of all things.
3. Socrates: Although not much is mentioned about him in the provided context, he is mentioned as a significant figure in the history of ancient Greek philosophy.
4. Plato: Founded the Academy, a model for higher education and scholarly research.
5. Aristotle: Founded the Lyceum, another model for higher education and scholarly research.

These individuals are mentioned as influential thinkers in the development of ancient Greek philosophy and its impact on Western intellectual traditions.


In [15]:
query = 'Explain briefly about the ancient greek history of philosophy?'

result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

[최종 답변 결과]
The ancient Greek history of philosophy is a monumental chapter in human intellectual history. Greek philosophers and scientists laid the groundwork for diverse fields of study, promoting a culture of inquiry, reason, and intellectual rigor. Their contributions advanced contemporary understanding and provided enduring frameworks that continue to influence modern thought and practice.


In [20]:
query = 'Who are the key figures in the ancient greek history of philosophy?'
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

[최종 답변 결과]
Unfortunately, the provided context does not mention specific key figures in ancient Greek philosophy. It only mentions that "Greek philosophers and scientists" laid the groundwork for diverse fields of study, but it does not provide names or details about individual philosophers.


In [21]:
docs = retriever.invoke(query)
for d in docs:
    print(d.page_content)
    print("---")

Greek ideas about ethics, governance, and the role of individuals in society continue to inform modern ethical theories, political systems, and educational curricula.
Conclusion
The philosophical and scientific endeavors of ancient Greece represent a monumental chapter in human intellectual history. Greek philosophers and scientists laid the essential groundwork for diverse fields of study, promoting a culture of inquiry, reason, and intellectual rigor. Their contributions not only advanced contemporary understanding but also provided enduring frameworks that continue to influence modern thought and practice. By exploring the rich legacy of Greek philosophy and science, we gain insight into the enduring quest for knowledge and the pursuit of wisdom that characterizes human civilization.
---
Chapter 6: Economy and Trade in Ancient Greece
---
Greek ideas about ethics, governance, and the role of individuals in society continue to inform modern ethical theories, political systems, and edu

In [22]:
file_path = './test.txt'

found_sentences = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if 'Thales' in line:
            found_sentences.append(line.strip())

if found_sentences:
    print(f"Found {len(found_sentences)} sentence(s) containing 'Thales' in {file_path}:")
    for sentence in found_sentences:
        print(sentence)
else:
    print(f"No sentences containing 'Thales' found in {file_path}.")

Found 15 sentence(s) containing 'Thales' in ./test.txt:
Pre-Socratic Philosophers: Focused on natural phenomena and the origins of the cosmos, including Thales, Anaximander, and Heraclitus.
Thales of Miletus: Proposed that water was the fundamental substance (arche) underlying all matter.
Thales and Anaximander: Proposed early models of the solar system and celestial mechanics.
II. Thales was the first to offer a purely physical explanation of the
genius of Athens. Thales and his successors down to Democritus were
detractors of everything Hellenic. Thales and the rest, we are told,
Thales, of Miletus, an Ionian geometrician and astronomer, about whose
to Thales all things have come from water. That the earth is entirely
confused fancies of mythologising poets. Not that Thales was an
interferences with the course of Nature. For the rest, Thales made
and astronomy. We have seen that to Thales water, the all-embracing
were to them precisely what water had been to Thales, what air was to
r

In [23]:
query = 'Who are Thales in the ancient greek history of philosophy?'
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

[최종 답변 결과]
Thales is mentioned in the provided context as one of the successors down to Democritus, who were not exactly what we should call philosophers in any sense of the word.


In [26]:
query = input("질문을 입력하세요: ")
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

질문을 입력하세요: What is the achivement of Thales ?
[최종 답변 결과]
Thales was the first to offer a purely physical explanation of the world.


### **왜 key features 를 못 찾지? 문서에는 있는데?**
- chunk_size=1000으로 쪼개면 chunk 수가 엄청 많아지게 됨 (대략 총 원문 9600줄 → 수백~수천 개 chunk).
- k=2는 그 수많은 chunk 중 딱 2개만 가져옵니다. 확률적으로 플라톤/아리스토텔레스 상세 내용이 담긴 chunk가 상위 2개 안에 들 가능성이 매우 낮음
- 실제로 검색된 2개는 아마 1장 초반부의 "Greek philosophers and scientists laid the groundwork..." 같은 일반적인 요약 문장일 듯 --> 답변 내용과 정확히 일치
- Supabase의 documents 테이블에는 약 780개의 벡터가 저장되어 있고, k=2는 이 780개 중 **딱 2개(전체의 0.25%)** 만 가져오는 것으로 질문과 의미적으로 가장 가까운 상위 2개가 하필 챕터 1 서두의 일반적인 요약 문장이었고, 철학자 이름이 담긴 chunk(97~300줄 근처)는 780개 중 3등, 5등, 10등 등 어딘가에 있었지만 k=2 밖으로 밀려나서 아예 전달되지 않은 것으로, LLM 입장에서는 "context에 없으니 모른다"고 정직하게 답한 것임
- **해결방안** ---> retriever = db.as_retriever(search_kwargs={"k": 10}) 로 코드를 변경하라

In [76]:
# @title Rewrite-Retrieve-Read
# simply prompts the LLM to rewrite the user’s query before performing retrieval


# create retriever to retrieve 2 relevant documents
retriever = db.as_retriever(search_kwargs={"k": 3})

@chain
def qa(input_query):
    # 1. Retrieval: 질문과 관련된 문서 조각 가져오기(fetch relevant documents)
    docs = retriever.invoke(input_query)

    # 2. Prompt 조립: 질문과 컨텍스트 결합(format prompt)
    formatted = prompt.invoke({"context": docs, "question": input_query})

    # 3. Generation: Groq LLM을 통해 답변 생성(generate answer)
    answer = llm.invoke(formatted)
    return answer

result1 = qa.invoke("""Today I woke up and brushed my teeth, then I sat down to read the
 news. But then I forgot the food on the cooker. Who are some key figures in
 the ancient greek history of philosophy?""")

print(result1.content)

Unfortunately, the provided context does not mention any specific key figures in ancient Greek history of philosophy. However, it does mention that "Greek philosophers and scientists laid the essential groundwork for diverse fields of study, promoting a culture of inquiry, reason, and intellectual rigor." 

Given the context, it seems that the text is more focused on the overall impact and contributions of ancient Greek philosophers and scientists rather than highlighting specific individuals.


In [77]:
rewrite_prompt = ChatPromptTemplate.from_template("""Provide a better search
 query for web search engine to answer the given question, end the queries
 with ’**’. Question: {x} Answer:""")

def parse_rewriter_output(message):
    return message.content.strip('"').strip("**")

rewriter = rewrite_prompt | llm | parse_rewriter_output

@chain
def qa_rrr(input):
    # rewrite the query
    new_query = rewriter.invoke(input)
    # fetch relevant documents
    docs = retriever.invoke(input)
    # format prompt
    formatted = prompt.invoke({"context": docs, "question": input})
    # generate answer
    answer = llm.invoke(formatted)
    return answer

# run
result2 = qa_rrr.invoke("""Today I woke up and brushed my teeth, then I sat down to read
 the news. But then I forgot the food on the cooker. Who are some key
 figures in the ancient greek history of philosophy?""")


print(result2.content)

Unfortunately, the provided context does not mention any specific key figures in ancient Greek history of philosophy. However, it does mention that "Greek philosophers and scientists laid the essential groundwork for diverse fields of study, promoting a culture of inquiry, reason, and intellectual rigor." 

Given the context, it seems that the text is discussing the broader impact and contributions of ancient Greek philosophers and scientists, rather than highlighting specific individuals.


In [86]:
# @title LCEL, No RAG
message_runnable = rewrite_prompt | llm
example_input = {"x": "Who are some key figures in ancient Greek philosophy?"}
response = message_runnable.invoke(example_input)
print(response.content)

"Key figures in ancient Greek philosophy" 

Optimized Search Query: 
"Socrates, Plato, Aristotle"


Groq 대신 Google Gemini API를 쓰도록 바꾼 코드 :
- langchain_google_genai 패키지의 ChatGoogleGenerativeAI를 사용
- userdata에서 Gemini API 키를 가져오는 부분만 추가
- Gemini 임베딩 모델 설정 (384차원으로 축소) : GoogleGenerativeAIEmbeddings

In [17]:
!pip install -q langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.0 MB/s eta 0:00:00


In [19]:
import os
from google.colab import userdata
from supabase.client import create_client, Client
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI  # Groq -> Gemini로 변경
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# ==========================================
# 1단계: 환경 변수 및 인증 정보 설정
# ==========================================
# 1) Supabase 접속 정보
SUPABASE_URL = "https://tllrshlupjcikzaywaqm.supabase.co"
SUPABASE_KEY = userdata.get('supabase')

supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 2) Google Gemini API 키 설정
# Colab의 Secrets(userdata)에 이름으로 API 키를 등록
GOOGLE_API_KEY = userdata.get('googleapikey')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# ==========================================
# 2단계: 데이터 로드, 분할 및 임베딩 (Ingestion)
# ==========================================
raw_documents = TextLoader('./test.txt', encoding='utf-8').load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

# Gemini 임베딩 모델 설정 (384차원으로 축소)
# text-embedding-004 -> gemini-embedding-001로 교체 (004는 2026.1.14 종료됨)
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=384
)

db = SupabaseVectorStore.from_documents(
    documents=documents,
    embedding=embeddings_model,
    client=supabase_client,
    table_name="documents",
    query_name="match_documents"
)

# ==========================================
# 3단계: 검색기(Retriever) 및 프롬프트, Gemini LLM 설정
# ==========================================
retriever = db.as_retriever(search_kwargs={"k": 2})

query = 'Who are the key figures in the ancient greek history of philosophy?'

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}

    Question: {question}"""
)

# Gemini 모델로 교체 (gemini-1.5-flash: 속도/비용 均衡, gemini-1.5-pro: 고성능)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

# ==========================================
# 4단계: LCEL @chain 데코레이터를 활용한 파이썬 함수 캡슐화 (Generation)
# ==========================================
print("Running RAG system using Gemini and Supabase...\n")

@chain
def qa_chain(input_query):
    docs = retriever.invoke(input_query)
    formatted = prompt.invoke({"context": docs, "question": input_query})
    answer = llm.invoke(formatted)
    return answer

result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 18.806613458s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 